# Jour 2 · Prévoir avec Prophet


## Objectifs

- mettre les données au format attendu par Prophet
- entraîner un modèle avec saisonnalités journalière et hebdomadaire
- évaluer et visualiser une prévision future

Prophet représente la série comme une combinaison de tendance, saisonnalités et effets spéciaux. Nous l'utilisons comme outil pratique, sans entrer dans sa formulation mathématique.

Prophet attend deux colonnes : `ds` pour la date et `y` pour la valeur. Les timestamps doivent être sans information de fuseau ; nous retirons donc le fuseau **après** avoir normalisé les données en UTC.

![Transformation d'un tableau ds et y en prévisions ds et yhat avec Prophet](../assets/jour_02/02_format_prophet_ds_y.png)

*Prophet apprend sur les couples date-valeur puis renvoie une estimation `yhat` pour les dates futures demandées.*

![Tendance, saisonnalités journalière et hebdomadaire combinées dans une prévision](../assets/jour_02/02_composantes_prophet.png)

*La représentation additive aide à comprendre ce que le modèle attribue au niveau général et aux motifs répétés.*

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_clean.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
hourly = df.set_index("timestamp")["power_kw"].resample("1h").mean()

cutoff = hourly.index.max() - pd.Timedelta(days=7)
train = hourly.loc[hourly.index <= cutoff]
test = hourly.loc[hourly.index > cutoff]

prophet_train = train.rename_axis("ds").rename("y").reset_index()
prophet_train["ds"] = prophet_train["ds"].dt.tz_localize(None)
prophet_train.head()

In [ ]:
model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=False,
    uncertainty_samples=0,
)
model.fit(prophet_train)

In [ ]:
future = pd.DataFrame({"ds": test.index.tz_localize(None)})
forecast = model.predict(future).set_index("ds")
predicted = pd.Series(forecast["yhat"].to_numpy(), index=test.index, name="Prophet")

mae = mean_absolute_error(test, predicted)
rmse = math.sqrt(mean_squared_error(test, predicted))
print(f"MAE  : {mae:.3f} kW")
print(f"RMSE : {rmse:.3f} kW")

![Passé d'entraînement, futur réel et prévision Prophet séparés par une frontière temporelle](../assets/jour_02/02_prevision_prophet.png)

*La zone de test représente un futur que le modèle n'a pas vu pendant son entraînement.*

In [ ]:
ax = test.plot(figsize=(13, 4), color="black", label="réel")
predicted.plot(ax=ax, label="Prophet", color="tab:blue")
ax.set_title("Prévision Prophet — puissance électrique")
ax.set_ylabel("kW")
ax.legend()
plt.show()

model.plot_components(model.predict(future))
plt.show()

### À vous de jouer — inspecter les plus grandes erreurs

Créez un tableau réel, prévision, erreur signée et erreur absolue. Affichez les dix heures les moins bien prévues.

**Indice :** Utilisez nlargest(10, 'absolute_error').

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — comparer à la baseline

Reconstruisez la baseline saisonnière hebdomadaire et comparez sa MAE à celle de Prophet.

In [ ]:
# Écrivez votre code ici.
pass

## À retenir

Prophet ne dispense ni d'une baseline ni d'un jeu de test temporel. Les composantes aident à expliquer le modèle, mais la métrique sur le futur décide s'il est utile.